# Stage 5 — Detect tissue without masks

**The question:** can we find the tissue using only the image, as well as the
expert annotations do?

This is the stage that makes the pipeline submittable. The previous attempt
chose where to look by clustering `train_label_masks`, and those files do not
exist for test slides — so that code could never run at submission time. From
here on, anything that decides *where to look* sees the image and nothing else.

The masks do appear in this notebook, once, at the very end, to score how well
the detector did. That is a different thing: masks may inform **what we learn**,
never **what we look at**. The scoring function stays in the notebook and never
travels into the tiling code.

**How the numbers get chosen.** Every threshold here comes out of a picture you
looked at, not a plausible-sounding value. The notebook is laid out so the
diagnostics come first and the decision comes last: build a display function,
look at both hospitals, sweep the threshold, compare fixed against automatic,
look at the extremes, then choose. The constants near the top are starting
points to make the first grid render — the cell at the end is where the real
choice gets recorded.

In [ ]:
"""Imports.

The first block makes the project's src/ folder importable. Locally it sits
one folder up from notebooks/. On Kaggle it arrives as an attached dataset, so
every attached dataset is checked for a src/ folder, and the notebook stops
with a clear message if there is none.

os         builds file paths.
cv2        (OpenCV) does the colour conversion, the automatic threshold, the
           clean-up of the mask and the search for connected blobs.
numpy      holds images and masks as arrays.
openslide  opens the slide and mask .tiff files.
pandas     loads the inventory and colour statistics, and holds the results.
matplotlib draws every grid and histogram.
tqdm       draws progress bars for the two sampling loops.
src.paths  where the data is, locally and on Kaggle.
src.tiling the project's tissue detector, checked against this notebook's at
           the end.
"""

import glob
import os
import sys

SRC_ROOT = next((root for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
                 if os.path.isfile(os.path.join(root, "src", "tiling.py"))), None)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

import cv2
import numpy as np
import openslide
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from src import tiling
from src.paths import IMAGE_DIR, MASK_DIR, find_input

In [ ]:
"""Paths and starting settings.

This notebook writes no files. Its output is the detector itself and the
settings in the decision cell at the end. The project's copy of both is in
src/tiling.py, and section 10 checks the two are the same.

Paths
-----
1. IMAGE_DIR, MASK_DIR and find_input come from src/paths.py, which knows
   where things are locally and on Kaggle. IMAGE_DIR holds the slide images;
   MASK_DIR holds the label masks, used only in section 8 to score the
   detector. find_input finds a file made by an earlier notebook, and stops
   with a clear error if it is nowhere.
2. The two inputs: the Stage 1 inventory and the Stage 2 colour statistics.
   Both are used only to choose which slides to look at, never by the
   detector itself.

Starting settings, not decisions
--------------------------------
These exist to make the first grids render. The decision cell at the end
reassigns the same names with the values actually chosen, after looking at the
pictures, and every function reads them when it runs.

3. SEED fixes which slides the random samples pick.
4. SAT_THRESHOLD is the saturation cut. Saturation is "how much colour", from
   0 to 255: empty glass is nearly white, so its saturation is near zero,
   while stained tissue is not. An automatic per-slide cut over 80 slides came
   out around 58 on Karolinska and 42 on Radboud, so a single fixed cut has to
   sit below both to avoid dropping faint Radboud tissue.
5. USE_OTSU switches from the fixed cut to an automatic one per slide.
6. OTSU_MIN and OTSU_MAX clamp the automatic cut. It picks a cut from each
   slide's own histogram, which adapts to a pale slide; but on a slide that is
   almost entirely background there is nothing to separate and it returns
   nonsense, so its result is kept inside this band.
7. MIN_COMPONENT_AREA: blobs smaller than this, measured in pixels at the
   smallest pyramid level, are dust rather than tissue.
8. INK_HUE_LO, INK_HUE_HI and INK_SAT_MIN describe pen ink, from Stage 2.
   OpenCV puts hue on a 0-179 wheel rather than 0-359, so hue numbers quoted
   elsewhere need halving. Stained tissue sits in the pink-to-purple part of
   the wheel; pen ink is strongly coloured and green-to-blue, which is this
   window.
"""

INVENTORY_PATH = find_input("slide_inventory.parquet")
COLOUR_STATS_PATH = find_input("slide_colour_stats.parquet")

SEED = 0

SAT_THRESHOLD = 15
USE_OTSU = False
OTSU_MIN, OTSU_MAX = 20, 60
MIN_COMPONENT_AREA = 40
INK_HUE_LO, INK_HUE_HI = 30, 135
INK_SAT_MIN = 60

In [ ]:
"""Load the inventory and attach each slide's colour statistics.

1. Read the Stage 1 inventory and the Stage 2 colour statistics.
2. Join them on slide_id. how="left" keeps every inventory slide.
3. Check that every slide received its colour numbers, since the sections below
   rank slides by them.
4. Print the table's shape and show the first rows.
"""

inventory = pd.read_parquet(INVENTORY_PATH)
colour_stats = pd.read_parquet(COLOUR_STATS_PATH)
inventory = inventory.merge(colour_stats, on="slide_id", how="left")
assert inventory["saturation_mean"].notna().all(), "a slide has no colour statistics"

print(inventory.shape)
inventory.head(3)

## 1. Reading a slide

Two helpers. Neither knows anything about the slide except its pixels.

`read_region` hands back four channels, the fourth being transparency, and
anything outside the slide comes back fully transparent with its colour set to
black. Calling `.convert("RGB")` would drop the transparency and leave those
areas **black**, which the detector would read as very dark tissue. Compositing
onto white is correct, because the empty part of a glass slide is white under a
scanner.

At the smallest pyramid level nothing falls outside the slide, so this does not
bite here — but the same function gets used at tile scale in Stage 6, where it
does, and having one correct version is the point.

In [ ]:
"""Three readers: two for slide images, one for label masks.

Slide images and label masks are both .tiff files, but they are different kinds
of thing. A slide pixel holds a colour; a mask pixel holds a class number from
0 to 5 in its red channel. So slides are composited onto white, and masks are
never composited: their raw red channel is the answer.
"""


def rgba_to_rgb_white(region):
    """Turn the RGBA image from read_region into RGB, with transparency white.

    1. Convert the image to an array with four channels: red, green, blue and
       alpha (opacity).
    2. Split off the colour channels, and scale alpha from 0-255 to 0-1.
    3. Blend each pixel with white in proportion to its opacity: fully opaque
       pixels keep their colour, fully transparent ones become white.
    4. Return the blend as whole-number RGB.
    """
    rgba = np.asarray(region)
    rgb = rgba[..., :3].astype(np.uint8)
    alpha = rgba[..., 3:4].astype(np.float32) / 255.0
    white = np.full_like(rgb, 255)
    return (rgb * alpha + white * (1.0 - alpha)).astype(np.uint8)


def read_smallest_level(slide_id):
    """Read a slide's smallest pyramid level as an RGB array.

    1. Open the slide's .tiff.
    2. Read its smallest level in full, the only level small enough to read
       whole.
    3. Close the slide in a finally block, so it is closed even if the read
       fails. A loop over thousands of slides that leaves files open dies with
       an error that does not name the cause.
    4. Composite onto white and return.
    """
    slide = openslide.OpenSlide(os.path.join(IMAGE_DIR, f"{slide_id}.tiff"))
    try:
        level = slide.level_count - 1
        region = slide.read_region((0, 0), level, slide.level_dimensions[level])
    finally:
        slide.close()
    return rgba_to_rgb_white(region)


def read_mask_labels(slide_id):
    """Read a label mask's smallest pyramid level as an array of class numbers.

    1. Open the mask's .tiff.
    2. Read its smallest level in full, which matches the slide's smallest level
       pixel for pixel.
    3. Close it in a finally block.
    4. Keep only the red channel, which holds the class number. Green and blue
       are always 0. No compositing: a class number blended with white would
       stop being a class number.
    """
    slide = openslide.OpenSlide(os.path.join(MASK_DIR, f"{slide_id}_mask.tiff"))
    try:
        level = slide.level_count - 1
        region = slide.read_region((0, 0), level, slide.level_dimensions[level])
    finally:
        slide.close()
    return np.asarray(region)[..., 0]

## 2. The detector

The whole thing in one function. This version keeps the exploration switches
(the automatic cut, the parts it returns for drawing); the project's version,
at the settings decided in section 9, is in `src/tiling.py`, and section 10
checks the two give the same mask.

**Why saturation.** Red-green-blue describes a colour the way a screen makes it,
which is awkward for "is this tissue?" — pale pink and dark purple are both
tissue but look nothing alike in RGB. HSV splits a colour into which colour it
is (hue), how strong it is (saturation), and how bright it is (value). Empty
glass is bright and colourless: high value, near-zero saturation. Stained tissue
has colour. So a saturation cut separates them cleanly.

**Why the cleanup.** A raw threshold gives a speckly answer: stray true pixels
in the background, small holes inside tissue. *Opening* (shrink then grow)
removes isolated specks. *Closing* (grow then shrink) fills small holes. Then
connected components — blobs of touching pixels — lets us drop anything too
small to be tissue.

**Why ink is subtracted.** Pen sits in the green-to-blue part of the colour
wheel, where stained tissue never does, and it is strongly coloured, so a plain
saturation cut calls it tissue. It must not count as tissue, or a tile that is
mostly pen stroke outranks a tile of real glands. Note what this does *not* do:
it does not discard the tile. A tile holding tissue and ink still has tissue and
still gets chosen. Stage 2 measured that a median 61% of ink sits on or against
tissue, so throwing those tiles away would delete real biopsy — and delete it
only in training, since test slides carry no ink at all.

In [ ]:
"""The detector, in one function, with the switches this notebook explores.
src/tiling.py holds the project's version at the decided settings."""


def tissue_mask(rgb, sat_threshold=None, min_area=None,
                use_otsu=None, return_parts=False):
    """Boolean tissue mask from an RGB array. Uses the image and nothing else.

    Args:
        rgb:           uint8 array [h, w, 3], RGB order
        sat_threshold: saturation cut, ignored when use_otsu is True
        min_area:      drop connected blobs smaller than this many pixels
        use_otsu:      pick the threshold per slide from its own histogram
        return_parts:  also return the ink mask and the threshold used

    Returns:
        tissue: boolean array [h, w]

    Steps
    1. Any setting left as None is read from the constants now, when the
       function runs. Writing sat_threshold=SAT_THRESHOLD in the signature
       instead would freeze the value when this cell runs, so editing the
       constant later would silently change nothing.
    2. Convert RGB to HSV: hue (which colour), saturation (how strong) and
       value (how bright). COLOR_RGB2HSV, not BGR2HSV: these arrays come from
       PIL, which stores red first. Getting it backwards swaps red and blue and
       quietly spoils every threshold below.
    3. Choose the saturation cut: either the fixed setting, or Otsu's method,
       which picks the cut that best splits this slide's own saturation values
       into two groups, clamped into the OTSU_MIN to OTSU_MAX band.
    4. coloured marks every pixel above the cut.
    5. ink marks pen: strongly coloured pixels in the green-to-blue part of the
       wheel. They are removed from the tissue, because a tile full of pen
       stroke must not outrank a tile of real glands.
    6. Combine the two and convert to 0/1 bytes; the clean-up functions need
       uint8, not True/False.
    7. Opening, a shrink then a grow, removes isolated specks. Closing, a grow
       then a shrink, fills small holes inside tissue. Both use a 3x3 square.
    8. connectedComponentsWithStats labels every blob of touching pixels and
       measures its area. Blob 0 is always the background. Keep only blobs of
       at least min_area pixels; smaller ones are dust.
    9. keep[labels] turns the per-blob decision back into a per-pixel mask.
    10. Return the mask, plus the ink mask and the cut used if asked.
    """
    sat_threshold = SAT_THRESHOLD if sat_threshold is None else sat_threshold
    min_area = MIN_COMPONENT_AREA if min_area is None else min_area
    use_otsu = USE_OTSU if use_otsu is None else use_otsu

    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    hue, saturation, value = hsv[..., 0], hsv[..., 1], hsv[..., 2]

    if use_otsu:
        threshold, _ = cv2.threshold(saturation, 0, 255,
                                     cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        threshold = float(np.clip(threshold, OTSU_MIN, OTSU_MAX))
    else:
        threshold = float(sat_threshold)

    coloured = saturation > threshold

    ink = (saturation > INK_SAT_MIN) & (hue > INK_HUE_LO) & (hue < INK_HUE_HI)

    mask = (coloured & ~ink).astype(np.uint8)

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    keep = np.zeros(count, dtype=bool)
    for i in range(1, count):
        keep[i] = stats[i, cv2.CC_STAT_AREA] >= min_area
    tissue = keep[labels]

    if return_parts:
        return tissue, ink, threshold
    return tissue

## 3. The display function, built before any tuning

Three panels per slide: the original, the mask, and the tissue kept on white.
The third panel is the one to read — it shows what the encoder would actually
be handed.

Building this first is deliberate. Tuning a threshold against a number instead
of a picture is how you end up with a detector that scores well and drops the
faint half of one hospital.

In [ ]:
"""Two helpers for looking: one draws detections, one picks slides."""


def show_detection(slide_ids, titles=None, use_otsu=None, sat_threshold=None,
                   suptitle=None):
    """Three panels per slide: the original, the mask, and the tissue kept.

    1. Captions default to the first 8 characters of each slide id.
    2. Make one row of three panels per slide. np.atleast_2d keeps the panel
       grid two-dimensional even when there is only one slide, and every frame
       is switched off.
    3. For each slide: read it, run the detector, and paint everything the
       detector did not keep white. The third panel is the one to read: it is
       what the encoder would actually be given.
    4. Caption the panels with the cut used, the share of the frame kept, and
       the share of the frame that was ink.
    """
    slide_ids = list(slide_ids)
    titles = list(titles) if titles is not None else [s[:8] for s in slide_ids]

    fig, axes = plt.subplots(len(slide_ids), 3,
                             figsize=(7.5, 2.7 * len(slide_ids)))
    axes = np.atleast_2d(axes)
    for ax in axes.ravel():
        ax.set_axis_off()

    for row, (slide_id, title) in enumerate(zip(slide_ids, titles)):
        rgb = read_smallest_level(slide_id)
        tissue, ink, threshold = tissue_mask(
            rgb, sat_threshold=sat_threshold, use_otsu=use_otsu, return_parts=True)

        kept = np.where(tissue[..., None], rgb, 255).astype(np.uint8)
        coverage = tissue.mean()

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f"{title}\nsat cut {threshold:.0f}", fontsize=7)
        axes[row, 1].imshow(tissue, cmap="gray")
        axes[row, 1].set_title(f"mask — {coverage:.1%} of frame\nink {ink.mean():.2%}",
                               fontsize=7)
        axes[row, 2].imshow(kept)
        axes[row, 2].set_title("tissue kept", fontsize=7)

    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    fig.tight_layout()
    return fig


def sample_slides(rows, n, seed=SEED, **filters):
    """A repeatable random sample of n rows, after filtering.

    Each keyword filter keeps rows where that column equals that value, for
    example provider="radboud". Returns every row if fewer than n remain.
    """
    for column, value in filters.items():
        rows = rows[rows[column] == value]
    return rows.sample(min(n, len(rows)), random_state=seed)

## 4. Look at both hospitals, separately

Stage 2 measured median saturation over the non-blank pixels at 88 on
Karolinska and 65 on Radboud. The two are a long way apart before anything is
tuned, so a cut that looks right on one may quietly fail on the other. Look at
them apart, never mixed together.

What to look for in the third panel: tissue that has gone missing at the faint
edges of a fragment, and background that has been kept.

In [ ]:
"""Six random Radboud slides through the detector, at the current setting."""

radboud = sample_slides(inventory, 6, provider="radboud")
show_detection(radboud["slide_id"],
               titles=[f"{r.slide_id[:8]}  ISUP {r.isup_grade}" for r in radboud.itertuples()],
               suptitle=f"Radboud — fixed saturation cut {SAT_THRESHOLD}")
plt.show()

In [ ]:
"""Six random Karolinska slides through the detector, at the current setting."""

karolinska = sample_slides(inventory, 6, provider="karolinska")
show_detection(karolinska["slide_id"],
               titles=[f"{r.slide_id[:8]}  ISUP {r.isup_grade}" for r in karolinska.itertuples()],
               suptitle=f"Karolinska — fixed saturation cut {SAT_THRESHOLD}")
plt.show()

## 5. Sweep the threshold on one slide from each hospital

The same slide at several cuts, side by side. This is what picks the number.

Read it for the failure that matters: **missed tissue is expensive, kept
background is cheap**. A cancerous region never tiled is a grade that can never
be predicted; a few extra background tiles waste a little compute. So when the
grids disagree, err downward.

In [ ]:
"""The same slide at six different cuts, side by side.

For each hospital, sweep one typical slide and the faintest slide in the
dataset, the one with the lowest mean saturation, since faint slides are where
a cut that is too high loses tissue first.
"""


def sweep_thresholds(slide_id, thresholds=(15, 20, 25, 30, 40, 50), suptitle=None):
    """Draw the detector's result at each cut: mask on top, tissue kept below.

    1. Read the slide once.
    2. For each cut, run the detector with that cut and draw the mask, captioned
       with the cut and the share of the frame kept.
    3. Underneath, draw the tissue kept, with everything else painted white.
    """
    rgb = read_smallest_level(slide_id)
    fig, axes = plt.subplots(2, len(thresholds), figsize=(2.1 * len(thresholds), 5.6))
    for ax in axes.ravel():
        ax.set_axis_off()
    for col, threshold in enumerate(thresholds):
        tissue = tissue_mask(rgb, sat_threshold=threshold)
        axes[0, col].imshow(tissue, cmap="gray")
        axes[0, col].set_title(f"cut {threshold}\n{tissue.mean():.1%}", fontsize=7)
        axes[1, col].imshow(np.where(tissue[..., None], rgb, 255).astype(np.uint8))
    fig.suptitle(suptitle or slide_id[:8], fontsize=11)
    fig.tight_layout()
    return fig


for provider in ("radboud", "karolinska"):
    faint = (inventory[inventory["provider"] == provider]
             .nsmallest(1, "saturation_mean").iloc[0])
    typical = sample_slides(inventory, 1, provider=provider).iloc[0]
    sweep_thresholds(typical.slide_id, suptitle=f"{provider} — typical slide")
    sweep_thresholds(faint.slide_id, suptitle=f"{provider} — faintest slide in the dataset")
plt.show()

## 6. Fixed cut against per-slide automatic

Otsu looks at one slide's own histogram and picks the split that best separates
it into two groups. That adapts to a pale slide, which a fixed number cannot.
It also fails badly on a slide that is almost entirely background, because then
there are no two groups to separate and it will happily cut the background in
half — which is why the result is clamped into a band.

Compare them on the same slides. The question is not which is cleaner on a good
slide; both will be. It is which one survives the bad ones.

In [ ]:
"""The hard cases through both methods, fixed cut first, then automatic.

The six hard cases:
- the 2 most nearly blank slides in the dataset;
- the 2 faintest Radboud slides;
- the 2 slides with the heaviest pen marks.
"""

awkward = pd.concat([
    inventory.nlargest(2, "percentage_empty"),
    inventory[inventory["provider"] == "radboud"].nsmallest(2, "saturation_mean"),
    inventory.nlargest(2, "ink_frac"),
])

show_detection(awkward["slide_id"],
               titles=[f"{r.slide_id[:8]} {r.provider[:4]}\n{r.percentage_empty:.0f}% blank"
                       for r in awkward.itertuples()],
               suptitle=f"Hard cases — fixed cut {SAT_THRESHOLD}")
show_detection(awkward["slide_id"],
               titles=[f"{r.slide_id[:8]} {r.provider[:4]}\n{r.percentage_empty:.0f}% blank"
                       for r in awkward.itertuples()],
               use_otsu=True, suptitle="Hard cases — per-slide automatic (Otsu, clamped)")
plt.show()

## 7. Coverage across many slides, per hospital

Now stop looking at individual slides and check the shape of the whole thing.
Run the detector over a sample from each hospital and compare how much of the
frame it keeps.

What would be worrying: the two hospitals' distributions sitting in different
places, which would mean the detector is systematically stricter on one of
them. Some difference is expected — Karolinska's slides are physically about
four and a half times larger, so the same tissue occupies a smaller share of
the frame.

In [ ]:
"""Run both methods over 200 slides per hospital and measure what they keep.

1. For each hospital, take a repeatable sample of COVERAGE_SAMPLE slides.
2. For each slide, read its smallest level and note that level's downsample
   factor, i.e. how many times smaller it is than full resolution.
3. Work out how much real tissue one pixel at this level covers: the
   downsample factor times the slide's microns per pixel gives microns per
   pixel at this level; squared and divided by a million, that is square
   millimetres per pixel. This is what makes the two hospitals comparable:
   without it, only pixel counts and shares are available, and those depend on
   slide size.
4. Record the frame's area in square millimetres.
5. Run the fixed and the automatic method and record, for each, the share of
   the frame kept and the tissue kept in square millimetres.
6. Collect everything in one table and print the medians per hospital.
"""

COVERAGE_SAMPLE = 200

coverage_rows = []
for provider in ("radboud", "karolinska"):
    rows = sample_slides(inventory, COVERAGE_SAMPLE, provider=provider)
    for r in tqdm(list(rows.itertuples()), desc=f"coverage {provider}"):
        slide = openslide.OpenSlide(os.path.join(IMAGE_DIR, f"{r.slide_id}.tiff"))
        try:
            level = slide.level_count - 1
            downsample = slide.level_downsamples[level]
            region = slide.read_region((0, 0), level, slide.level_dimensions[level])
        finally:
            slide.close()
        rgb = rgba_to_rgb_white(region)

        microns_per_pixel = downsample * float(r.tiff_mpx[0])
        mm2_per_pixel = (microns_per_pixel ** 2) / 1e6

        row = {"slide_id": r.slide_id, "provider": provider, "isup_grade": r.isup_grade,
               "frame_mm2": rgb[..., 0].size * mm2_per_pixel}
        for name, kwargs in (("fixed", {}), ("otsu", {"use_otsu": True})):
            found = tissue_mask(rgb, **kwargs)
            row[f"{name}_share"] = found.mean()
            row[f"{name}_mm2"] = found.sum() * mm2_per_pixel
        coverage_rows.append(row)

coverage = pd.DataFrame(coverage_rows)
print(coverage.groupby("provider")[["fixed_share", "fixed_mm2", "frame_mm2"]]
      .median().round(3).to_string())

In [ ]:
"""The same detector, measured two ways, then the thin end.

Read the two histograms together. The left one, share of the frame kept,
separates the hospitals almost completely and looks like a detector that is far
stricter on Karolinska. It is not. Share of frame is tissue divided by frame
size, and Karolinska frames are about four times larger, so the same amount of
tissue gives a much smaller share. The right one, tissue in square millimetres,
divides by nothing, and the two hospitals sit on top of each other. Whenever a
per-slide number is compared across hospitals, check first whether it is a
ratio with slide size underneath.

1. Draw both histograms, one colour per hospital.
2. Print the full summary of share, tissue area and frame area per hospital.
3. Count the slides where each method found less than 1 mm² of tissue, and
   which hospital they came from. These are the slides Stage 6 will struggle
   to get tiles from.
"""

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, column, label in ((axes[0], "fixed_share", "share of the frame"),
                          (axes[1], "fixed_mm2", "tissue found (mm²)")):
    for provider, rows in coverage.groupby("provider"):
        ax.hist(rows[column], bins=40, alpha=0.55, label=provider)
    ax.set_xlabel(label)
    ax.legend()
axes[0].set_ylabel("slides")
fig.suptitle("The same detector, measured two ways", fontsize=12)
fig.tight_layout()
plt.show()

print(coverage.groupby("provider")[["fixed_share", "fixed_mm2", "frame_mm2"]]
      .describe(percentiles=[.05, .5, .95]).round(3).T.to_string())

print("\nslides with less than 1 mm² of tissue found:")
for column in ("fixed_mm2", "otsu_mm2"):
    thin = coverage[coverage[column] < 1.0]
    print(f"  {column}: {len(thin)} of {len(coverage)}",
          thin["provider"].value_counts().to_dict())

## 8. Score the detector against the label masks

This is the one place in the project where masks are allowed, and it is worth
being precise about why it is allowed. The rule is that masks may inform **what
we learn**, never **what we look at**. Scoring a detector is the first: the
score is read by us, in this notebook, and the detector itself still runs on
pixels alone. Nothing here travels into the tiling code.

**Recall, not overlap.** The two errors are not equally expensive. Tissue the
detector missed is a region that never becomes a tile, so a cancerous area there
is a grade we can never predict. Background it wrongly kept just wastes a few
tiles. So the number to maximise is the share of annotated tissue we found — a
detector that finds 99% of the tissue plus some background beats one that finds
92% cleanly.

**And not all annotated tissue is equally worth finding.** The masks label
stroma — the pale connective tissue between the glands — as tissue, and stroma
has very little colour, so a saturation cut misses a lot of it. That drags the
overall recall down while costing nothing that matters. Measured here, recall
against *all* labels is about 0.80 on Radboud but recall against the
*cancerous* labels is about 0.99. Those are very different conclusions about
the same detector, and only the second one bears on the grade. The scoring
below reports both; the cancer figure is the one to read.

**One caveat when reading the result.** In the masks, 0 means background *or
unknown*, so real tissue is sometimes labelled 0. If the detector finds more
tissue than the mask does, that is not automatically wrong. Look at those cases
before tightening anything.

100 slides lack a mask entirely, all of them Radboud, so they are simply absent
from this check.

In [ ]:
"""Score both methods against the label masks, 100 slides per hospital.

1. Take a repeatable sample of slides that have a mask, per hospital. A
   different seed from the coverage sample, so the two checks see different
   slides.
2. For each slide, read the mask's class numbers and the slide itself. If
   either cannot be read, or the two are different sizes, record the slide as
   skipped rather than compare misaligned arrays.
3. annotated is every pixel with a class above 0, i.e. tissue of any kind.
4. cancer is the cancerous part. The hospitals code their masks differently:
   Radboud uses 3, 4 and 5 for cancerous epithelium, Karolinska uses 2 for
   cancerous tissue. Everything else above 0 is other tissue, mostly stroma.
5. Run both methods, and record for each:
   - recall: the share of all annotated tissue that the detector found;
   - recall_cancer: the share of annotated cancer it found. This is the number
     that matters: missed stroma costs a few tiles, missed cancer is a grade we
     can never predict. Slides whose mask contains no cancer get no value here;
   - precision: the share of what the detector kept that was annotated;
   - the share of the frame found, and the share annotated.
6. Collect the results in one table and report how many slides were skipped.
"""

MASK_SAMPLE = 100

score_rows = []
skipped = []
with_mask = inventory[inventory["mask_exists"]]
for provider in ("radboud", "karolinska"):
    rows = sample_slides(with_mask, MASK_SAMPLE, seed=SEED + 1, provider=provider)
    for r in tqdm(list(rows.itertuples()), desc=f"scoring {provider}"):
        try:
            labels = read_mask_labels(r.slide_id)
            rgb = read_smallest_level(r.slide_id)
        except Exception as error:
            skipped.append((r.slide_id, str(error)[:80]))
            continue
        if labels.shape != rgb.shape[:2]:
            skipped.append((r.slide_id, f"shape {labels.shape} vs {rgb.shape[:2]}"))
            continue

        annotated = labels > 0
        if provider == "radboud":
            cancer = (labels >= 3) & (labels <= 5)
        else:
            cancer = labels == 2

        for name, kwargs in (("fixed", {}), ("otsu", {"use_otsu": True})):
            found = tissue_mask(rgb, **kwargs)
            score_rows.append({
                "slide_id": r.slide_id,
                "provider": provider,
                "method": name,
                "recall": (found & annotated).sum() / max(annotated.sum(), 1),
                "recall_cancer": ((found & cancer).sum() / cancer.sum()
                                  if cancer.sum() else np.nan),
                "precision": (found & annotated).sum() / max(found.sum(), 1),
                "found_share": found.mean(),
                "annotated_share": annotated.mean(),
            })

scores = pd.DataFrame(score_rows)
print(f"{len(skipped)} slides skipped", skipped[:5] if skipped else "")

In [ ]:
"""Read the scores.

1. Median recall on all tissue, recall on cancer, and precision, per hospital
   and method. The gap between the first two is stroma the detector did not
   keep, which costs nothing that matters.
2. The full spread of cancer recall, including the 5th percentile, because the
   worst slides matter more than the typical one.
3. The 10 slides with the lowest cancer recall, the ones to look at next.
"""

print("median recall, all annotated tissue vs annotated cancer only")
print(scores.groupby(["provider", "method"])[["recall", "recall_cancer", "precision"]]
      .median().round(3).to_string())

print("\nrecall on cancer, full spread (this is the number that matters)")
print(scores.groupby(["provider", "method"])["recall_cancer"]
      .describe(percentiles=[.05, .25, .5]).round(3).to_string())

print("\nworst 10 slides by cancer recall (these are the ones to look at):")
worst = scores.dropna(subset=["recall_cancer"]).sort_values("recall_cancer").head(10)
print(worst[["slide_id", "provider", "method", "recall", "recall_cancer", "precision"]]
      .to_string(index=False))

In [ ]:
"""Look at the worst cases rather than accepting the number.

The five distinct slides with the lowest cancer recall, through the detector. A
low recall means either a detector that is too strict, or a mask that marks
something the image does not show; only the picture tells which.
"""

show_detection(worst["slide_id"].unique()[:5],
               titles=[f"{s[:8]}" for s in worst["slide_id"].unique()[:5]],
               suptitle="Lowest recall against the masks")
plt.show()

## 9. Decide

Fill in what you chose, having looked. This cell is the output of the stage —
everything above it is the argument. `src/tiling.py` holds the same values,
and section 10 checks they match.

If the fixed cut and the automatic one were close, prefer the fixed one: it is
predictable, it is one number to record in a manifest, and it cannot surprise
you on a slide you have not seen. Prefer automatic only if it clearly rescued
slides the fixed cut lost.

A provider-specific setting is a last resort. It would mean the tiling code
needs to know which hospital a slide came from, and at submission time that
information may not exist — check `test.csv` before depending on it.

In [ ]:
"""The decision.

These are the same names as the starting settings at the top of the notebook,
on purpose. This cell is the authority: run it, then re-run any grid above, and
the grid uses what was decided here. A second set of names would only let the
two drift apart. src/tiling.py holds the same values; section 10 checks them.

1. Set the chosen method, cut and minimum blob size.
2. Print the settings Stage 6 must use.
3. Print, for the chosen method, the median recall per hospital and the median
   tissue found per slide.
"""

USE_OTSU = False
SAT_THRESHOLD = 15
MIN_COMPONENT_AREA = 40

print("Tissue detection settings for Stage 6")
print(f"  method:              {'per-slide Otsu, clamped' if USE_OTSU else 'fixed saturation cut'}")
if USE_OTSU:
    print(f"  Otsu clamp:          {OTSU_MIN} to {OTSU_MAX}")
else:
    print(f"  saturation cut:      {SAT_THRESHOLD}")
print(f"  min component area:  {MIN_COMPONENT_AREA} px at the smallest level")
print(f"  ink excluded from tissue: hue {INK_HUE_LO}-{INK_HUE_HI}, saturation > {INK_SAT_MIN}")

final = scores[scores["method"] == ("otsu" if USE_OTSU else "fixed")]
print("\nmedian recall against the masks, by hospital:")
print(final.groupby("provider")[["recall", "recall_cancer"]].median().round(3).to_string())
print("  (recall_cancer is the one that matters; the gap is stroma we did not keep)")
print("\nmedian coverage per hospital:")
print(coverage.groupby("provider")[("otsu_mm2" if USE_OTSU else "fixed_mm2")]
      .median().round(3).to_string(), "mm² of tissue per slide")

## 10. The project's detector is this one

The detector decided here now lives in `src/tiling.py`, which is what the tile
selector, the feature extraction and the submission all use. This notebook
keeps its own copy because it needs the exploration switches (the automatic
cut, the parts it returns for drawing) that the project's version has no use
for.

So the two must be shown to be the same detector at the decided settings: the
same numbers, and the same mask pixel for pixel on many slides. If this check
passes, every result above holds for the project's detector.

In [ ]:
"""Compare this notebook's detector, at the decided settings, with src/tiling.py.

1. The settings must match: saturation cut, minimum blob size, and the ink
   window. The decision cell above must have been run, and must have chosen the
   fixed cut, since the project's detector has no automatic option.
2. For every slide in the coverage sample (200 per hospital), run both
   detectors on the same slide, each through its own reader, and require
   identical masks. This also checks that the two ways of compositing onto
   white agree.
"""

assert not USE_OTSU, "the project's detector uses the fixed cut only"
assert (SAT_THRESHOLD, MIN_COMPONENT_AREA) == (tiling.SATURATION_CUT, tiling.MIN_CC_AREA)
assert (INK_HUE_LO, INK_HUE_HI, INK_SAT_MIN) == (*tiling.INK_HUE_THRESH, tiling.INK_SAT_THRESH)

for slide_id in tqdm(coverage["slide_id"], desc="comparing detectors"):
    here = tissue_mask(read_smallest_level(slide_id))
    image, _ = tiling.read_smallest_level(os.path.join(IMAGE_DIR, f"{slide_id}.tiff"))
    project = tiling.tissue_mask(np.asarray(tiling.rgba_to_rgb_white(image)))
    assert np.array_equal(here, project), f"{slide_id}: detectors disagree"
print(f"identical masks on all {len(coverage)} slides: src/tiling.py is the detector decided here")

## What to record

Into `FINDINGS.md`:

- Which method and which numbers, and the pictures that decided them.
- Median recall against the masks, per hospital, and what the worst cases were.
- Median tissue coverage per hospital, and how many slides have under 1 mm² of tissue.
- Whether ink exclusion changed anything visible.

Then Stage 6, which places tiles using this mask and nothing else, through
`src/tiling.py`.

**The rule carried forward:** the tiling function reads the image file and
nothing else. No masks, no CSVs, no labels, no hospital. If it needs any of
them, it cannot be submitted.